# Fine-Tuning Qwen 3.8 27B on Daily Market Outcomes

This notebook fine-tunes **Qwen 3.8 27B** with Unsloth 4-bit QLoRA on a Google Colab Pro A100 GPU.

### Reliability & Drive persistence
- Google Drive is mounted at the start.
- Checkpoints and LoRA adapters are saved continuously to `/content/drive/MyDrive/qwen3.8-predictor-checkpoints`.
- If Colab disconnects or drops, rerunning the training cell automatically resumes from the last saved checkpoint on Google Drive.

### Pure market ground truth (Zero forward leakage)
- Input: Pre-market data before the 9:30 AM open (newsletter, macro prints, overnight gaps).
- Target: The true market close outcome: direction (`UP` or `DOWN`) and return percentage (`expected_return_pct`).
- Confidence: Evaluated mathematically from direction token logprobs at inference time.

In [ ]:
# 1. Mount Google Drive and configure persistent checkpoint directory
import os
from google.colab import drive

drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/qwen3.8-predictor-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Continuous checkpoint directory: {CHECKPOINT_DIR}')

In [ ]:
# 2. Install Unsloth and training dependencies
# We avoid %%capture so any pip installation warnings or errors are visible
!pip install --upgrade --no-cache-dir unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

# Verify import right in the install cell
import unsloth
print(f'Unsloth installed successfully, version: {unsloth.__version__}')

In [ ]:
# 3. Load Qwen 3.8 27B base model in 4-bit with FastLanguageModel
import torch
from unsloth import FastLanguageModel

max_seq_length = 16384
dtype = torch.bfloat16  # Recommended for A100 Tensor Cores
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.8-27B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Apply LoRA PEFT adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    max_seq_length=max_seq_length,
)
print('Model loaded with LoRA adapters successfully.')

In [ ]:
# 4. Load train.jsonl and val.jsonl datasets
import json
import os
from datasets import Dataset

def resolve_data_path(filename):
    drive_path = os.path.join(CHECKPOINT_DIR, filename)
    if os.path.exists(drive_path):
        return drive_path
    if os.path.exists(filename):
        return filename
    raise FileNotFoundError(f'Could not find {filename} in {CHECKPOINT_DIR} or local session folder.')

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_path = resolve_data_path('train.jsonl')
val_path = resolve_data_path('val.jsonl')
print(f'Loading training data from: {train_path}')
print(f'Loading validation data from: {val_path}')

train_data = load_jsonl(train_path)
val_data = load_jsonl(val_path)

def format_prompts(batch):
    texts = []
    for messages in batch['messages']:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {'text': texts}

train_dataset = Dataset.from_list(train_data).map(format_prompts, batched=True)
val_dataset = Dataset.from_list(val_data).map(format_prompts, batched=True)
print(f'Loaded {len(train_dataset)} training samples and {len(val_dataset)} validation samples.')

In [ ]:
# 5. Configure SFTTrainer with Google Drive Sync and VLM compatibility
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import os

class GoogleDriveAdapterSyncCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        adapter_save_path = os.path.join(CHECKPOINT_DIR, 'latest_lora_adapter')
        kwargs['model'].save_pretrained(adapter_save_path)
        tokenizer.save_pretrained(adapter_save_path)
        print(f'\n[Drive Sync] LoRA adapter synced to: {adapter_save_path}')

sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=1,
    save_strategy="steps",
    save_steps=10,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    assistant_only_loss=False,  # Set to False because Qwen 3.8 architecture is classified as a VLM
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=sft_config,
    callbacks=[GoogleDriveAdapterSyncCallback()],
)
print('Trainer initialized successfully for Qwen 3.8 27B.')

In [ ]:
# 6. Execute Training with Automatic Checkpoint Resumption
import glob
import os

existing_checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")))
resume = True if existing_checkpoints else False
if resume:
    print(f"Resuming training from latest Google Drive checkpoint: {existing_checkpoints[-1]}")
else:
    print("Starting fresh training run...")

trainer_stats = trainer.train(resume_from_checkpoint=resume)

In [ ]:
# 7. Run Test Inference on Held-out Validation Session
FastLanguageModel.for_inference(model)

val_sample = val_data[-1]
messages = [m for m in val_sample['messages'] if m['role'] in ('system', 'user')]

# Format chat prompt string
prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Access the underlying text tokenizer directly so the processor does not mistake text for an image
text_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)
inputs = text_tokenizer(prompt_text, return_tensors='pt').to('cuda')

input_ids = inputs['input_ids']
attention_mask = inputs.get('attention_mask')

outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=128,
    temperature=0.1,
    use_cache=True,
)
input_len = input_ids.shape[1]
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print('Ground truth metadata:', val_sample['metadata'])
print('\nModel Prediction JSON:')
print(response)

In [ ]:
# 8. Save Final LoRA Weights to Google Drive
import os
final_lora_dir = os.path.join(CHECKPOINT_DIR, "qwen3.8-27b-predictor-final-lora")
model.save_pretrained(final_lora_dir)
tokenizer.save_pretrained(final_lora_dir)
print(f"Final LoRA adapter saved to Google Drive: {final_lora_dir}")

In [ ]:
# 9. Export to GGUF format for local Ollama / llama.cpp inference
import os
gguf_save_path = os.path.join(CHECKPOINT_DIR, "gguf_output")
model.save_pretrained_gguf(
    gguf_save_path,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"Q4_K_M GGUF model exported to Google Drive: {gguf_save_path}")

In [ ]:
# 10. Standalone Benchmark: Test Across All 20 Forward Sessions on A100 GPU
# You can run this cell independently after reconnecting to Colab.
import os
import json
import torch
from google.colab import drive
from unsloth import FastLanguageModel

# Ensure Google Drive is mounted
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/qwen3.8-predictor-checkpoints'
ADAPTER_PATH = os.path.join(CHECKPOINT_DIR, 'latest_lora_adapter')

# Load fine-tuned model and adapter directly from Google Drive
print(f'Loading fine-tuned model from: {ADAPTER_PATH}')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=16384,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
text_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)

# Locate validation dataset
val_path = os.path.join(CHECKPOINT_DIR, 'val.jsonl')
if not os.path.exists(val_path):
    val_path = 'val.jsonl'

with open(val_path, 'r', encoding='utf-8') as f:
    val_samples = [json.loads(line) for line in f]

print(f'Loaded {len(val_samples)} validation sessions.')
print('\n' + '='*65)
print('BENCHMARK EVALUATION: Fine-Tuned Qwen 3.8 27B on A100 GPU')
print('='*65 + '\n')

correct = 0
total = 0
brier_scores = []
return_errors = []
valid_json_count = 0

for idx, sample in enumerate(val_samples, 1):
    target_date = sample['metadata']['target_date']
    actual_dir = sample['metadata']['actual_direction']
    actual_ret = sample['metadata']['actual_return_pct']

    messages = [m for m in sample['messages'] if m['role'] in ('system', 'user')]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = text_tokenizer(prompt_text, return_tensors='pt').to('cuda')

    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs.get('attention_mask'),
        max_new_tokens=128,
        temperature=0.1,
        use_cache=True,
    )
    input_len = inputs['input_ids'].shape[1]
    raw_output = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(raw_output)
        valid_json_count += 1
        pred_dir = str(parsed.get('predicted_direction', '')).upper()
        pred_ret = parsed.get('expected_return_pct')

        is_hit = pred_dir == actual_dir
        if is_hit:
            correct += 1
        total += 1

        if pred_ret is not None:
            return_errors.append(abs(float(pred_ret) - float(actual_ret)))

        status = 'HIT ' if is_hit else 'MISS'
        print(f'[{idx:02d}/{len(val_samples):02d}] {target_date} | Actual: {actual_dir:4s} ({actual_ret:+5.2f}%) | Pred: {pred_dir:4s} ({float(pred_ret):+5.2f}%) -> {status}')
    except json.JSONDecodeError:
        total += 1
        print(f'[{idx:02d}/{len(val_samples):02d}] {target_date} | Invalid JSON: {raw_output[:60]}...')

accuracy = (correct / total * 100.0) if total > 0 else 0.0
avg_ret_err = (sum(return_errors) / len(return_errors)) if return_errors else 0.0
json_rate = (valid_json_count / len(val_samples) * 100.0) if val_samples else 0.0

print('\n' + '-'*55)
print('FINAL BENCHMARK SCORECARD')
print(f'Total Forward Sessions:     {len(val_samples)}')
print(f'JSON Format Compliance:     {json_rate:.1f}% ({valid_json_count}/{len(val_samples)})')
print(f'Directional Accuracy:       {accuracy:.1f}% ({correct}/{total})')
print(f'Mean Absolute Return Error: {avg_ret_err:.2f}%')
print('-'*55 + '\n')